# Frente 3 — Pipeline de pré-processamento e reprodutibilidade

Integra as operações definidas nas Frentes 1 e 2, sem depender dos CSVs já tratados.

## 1. Importar bibliotecas e funções

In [1]:
import pandas as pd
from pathlib import Path
import sys

# Localiza a raiz do projeto quando o notebook é executado na raiz ou em Frente3.
raiz = Path.cwd()
if not (raiz / "Frente1").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "Frente1"))
sys.path.insert(0, str(raiz / "Frente2"))

from limpeza_pipeline import limpar_pedidos, limpar_itens, limpar_produtos, limpar_avaliacoes, limpar_clientes
from transformacoes_pipeline import montar_base, criar_atributos, transformar_dados, selecionar_atributos

## 2. Função principal `preprocessar()`

In [ ]:
# Integração das funções existentes das Frentes 1 e 2.
def preprocessar(pedidos, itens, produtos, avaliacoes, clientes, colunas_referencia=None):
    # Limpeza existente da Frente 1
    pedidos = limpar_pedidos(pedidos)
    itens = limpar_itens(itens)
    produtos = limpar_produtos(produtos)
    avaliacoes = limpar_avaliacoes(avaliacoes)
    clientes = limpar_clientes(clientes)

    # Transformações existentes da Frente 2
    base = montar_base(pedidos, itens, produtos, avaliacoes, clientes)
    base = criar_atributos(base)
    base = transformar_dados(base)
    tabela, selecao = selecionar_atributos(base)

    # Usa as colunas da tabela completa como referência para testar amostras
    if colunas_referencia is not None:
        for coluna in colunas_referencia:
            if coluna not in tabela.columns:
                if coluna in base.columns:
                    tabela[coluna] = base[coluna]
                else:
                    tabela[coluna] = 0
        tabela = tabela[colunas_referencia]

    return tabela

## 3. Carregar os cinco CSVs originais

In [3]:
pasta = raiz / "base_de_dados_originais"

pedidos = pd.read_csv(pasta / "olist_orders_dataset.csv")
itens = pd.read_csv(pasta / "olist_order_items_dataset.csv")
produtos = pd.read_csv(pasta / "olist_products_dataset.csv")
avaliacoes = pd.read_csv(pasta / "olist_order_reviews_dataset.csv")
clientes = pd.read_csv(pasta / "olist_customers_dataset.csv")

print("Dados originais carregados.")

Dados originais carregados.


## 4. Executar e validar o pipeline completo

In [4]:
tabela = preprocessar(pedidos, itens, produtos, avaliacoes, clientes)

print("Tamanho da tabela:", tabela.shape)
print("Pedidos únicos:", tabela["id_pedido"].is_unique)
print("Notas válidas:", tabela["nota_avaliacao"].between(1, 5).all())
tabela.head()

Tamanho da tabela: (98673, 36)
Pedidos únicos: True
Notas válidas: True


,id_pedido,nota_avaliacao,grupo_nota,nota_baixa,mes_compra,estado_cliente,categoria_produto,status_pedido,situacao_entrega,valor_produtos,...,valor_produtos_zscore,valor_frete_minmax,valor_frete_zscore,tempo_entrega_dias_minmax,tempo_entrega_dias_zscore,qtd_vendedores,entrega_antes_do_prazo,entrega_atrasada,entrega_no_prazo,entrega_sem_data
0,809a282bbd5dbcabb6f2f724fca862ec,1,baixa,1,2016-09,SP,NaN,canceled,sem_data,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1
1,bfbd0f9bdef84302105ad712db648a6c,1,baixa,1,2016-09,SP,beleza_saude,delivered,atrasada,134.97,...,-0.012241,0.004730,-0.662041,0.261188,4.470273,1.0,0,1,0,0
2,e5215415bb6f76fe3b7cb68103a0d1c0,1,baixa,1,2016-10,CE,NaN,canceled,sem_data,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1
3,e2144124f98f3bf46939bc5183104041,4,intermediaria,0,2016-10,RJ,bebes,delivered,antes_do_prazo,39.90,...,-0.466073,0.006106,-0.547801,0.038928,-0.411932,1.0,1,0,0,0
4,9aa3197e4887919fde0307fc23601d7a,4,intermediaria,0,2016-10,SP,fashion_bolsas_e_acessorios,delivered,antes_do_prazo,25.90,...,-0.532905,0.004858,-0.651404,0.051535,-0.135002,1.0,1,0,0,0


## 5. Testar com uma amostra relacionada

A amostra é escolhida a partir dos IDs de pedidos. Assim, os itens e avaliações correspondentes continuam disponíveis.

In [ ]:
ids = pedidos["order_id"].sample(100, random_state=42)

pedidos_amostra = pedidos[pedidos["order_id"].isin(ids)]
itens_amostra = itens[itens["order_id"].isin(ids)]
avaliacoes_amostra = avaliacoes[avaliacoes["order_id"].isin(ids)]
produtos_amostra = produtos[produtos["product_id"].isin(itens_amostra["product_id"])]
clientes_amostra = clientes[clientes["customer_id"].isin(pedidos_amostra["customer_id"])]

tabela_amostra = preprocessar(
    pedidos_amostra, itens_amostra, produtos_amostra,
    avaliacoes_amostra, clientes_amostra,
    colunas_referencia=tabela.columns
)
print("Amostra:", tabela_amostra.shape)
print("Mesmas colunas:", tabela_amostra.columns.equals(tabela.columns))
print("Pedidos únicos:", tabela_amostra["id_pedido"].is_unique)

Amostra: (99, 36)
Mesmas colunas: True
Pedidos únicos: True


In [ ]:
# Executar o pipeline novamente com os mesmos dados

tabela_teste = preprocessar(
    pedidos,
    itens,
    produtos,
    avaliacoes,
    clientes
)

# Comparar os resultados
pd.testing.assert_frame_equal(tabela, tabela_teste)

print("Teste de reprodutibilidade aprovado!")

## 6. Salvar a tabela analítica

In [6]:
saida = raiz / "tabela_analitica_final.csv"
tabela.to_csv(saida, index=False)
print("Arquivo salvo em:", saida)

Arquivo salvo em: c:\Users\NONATO\Desktop\Projeto-Analise-Exploratoria-Gerencia-Experiencia-Cliente\tabela_analitica_final.csv


**Observação técnica:** a Frente 2 recalcula os parâmetros de normalização a cada execução. Este notebook conserva os nomes e a ordem das colunas na amostra, mas ainda não congela as médias e os desvios do histórico. Caso isso seja exigido para comparações entre bases, será preciso ajustar essa parte em conjunto com a lógica da Frente 2.

**Uso de IA:** registrar a assistência efetivamente aproveitada em `uso_de_ia.md` e substituir `N` pelo número correto dos registros.